# 04 · Triage and approve

*Part 2, sections 5-6 and Part 3, sections 2 and 6.* With more reviewers and more traffic the manual
loop stops scaling, in four ways: **criteria drift**, **reviewer disagreement**, **duplicate
signal**, and **judge drift**. The bottleneck is reading.

Two jobs take it over, deliberately kept apart so the agent that proposes a test never approves it:

* **triage** reads the new failures since its checkpoint, sends contradictions to a person, and has
  an analyst group the rest into themes with one criterion each, proposing tests
* **approve** checks each proposed test and merges it, or sends it to the human queue

In [ ]:
import _nb as nb

experiment_id = nb.setup()

In [ ]:
import mlflow
from examples import reviewer
from selfimprove import agents, config, improvers, instructions, store, traces

drift = [q for q in open("examples/drift.txt").read().splitlines() if q and not q.startswith("#")]
regular = [q for q in open("examples/questions.txt").read().splitlines() if q and not q.startswith("#")]
for q in drift * 2 + regular:
    agents.run(q, source="production")
print(reviewer.review_all(sample=1.0))

A new kind of question has started arriving: escalations, discounts, roadmap requests. v2 never
mentions `support_process`. Meanwhile, the reviewers do not always agree:

In [ ]:
disputed = [t for t in traces.production() if len(set(traces.human_labels(t).values())) > 1]
print(len(disputed), "traces where reviewers disagree")
for t in traces.production()[:40]:
    labels = traces.human_labels(t)
    if labels:
        print(f"{str(labels):<32} {traces.question_of(t)[:60]}")

## 1. What the analyst reads

The failure report is the signal, compacted: input, output, every failing verdict with its source
and rationale, the tool calls, human failures first. Holdout cases are filtered out before
anything upstream of the gate sees them.

In [ ]:
holdout = store.holdout_questions()
visible = [t for t in traces.production() if not store.is_holdout(traces.question_of(t), holdout)]
report = improvers.failure_report(visible)
print(len(report), "failing traces in the report")
report[0]

## 2. Triage

`triage()` is the scheduled job from Part 3, section 2. It runs as an MLflow run, and logs its
proposal on that run (`proposal.json`) for the approver. With credentials, the analyst
(`improvers.analyze`, on the judge model) names the themes; without, the first failing verifier does.

In [ ]:
from selfimprove.workflows.triage import triage

result = triage(force=True)
result.metrics

In [ ]:
proposal = mlflow.artifacts.load_dict(f"runs:/{result.data['proposal_run']}/proposal.json")
for th in proposal["themes"]:
    print(f"{th['slug']:<28} x{th['count']:<3} {'(reviewer noise) ' if th['is_reviewer_noise'] else ''}{th['criterion'][:90]}")
print()
for r in proposal["records"][:6]:
    print(f"{r['tags'].get('theme', ''):<28} {r['inputs']['question'][:60]}")

## 3. Approve

A proposed test joins the dataset only if it **reproduces** the failure it came from, is **not a
near duplicate**, stays **under the theme cap** (`SI_TESTS_PER_THEME_CAP`), and passes a
**reviewer agent** (a separate prompt from the analyst). Anything else goes to the human queue.
Real themes with an approved test become registered `theme_<slug>` judges: the verifier grows
with the evidence.

In [ ]:
from selfimprove.workflows.approve import approve

approve().metrics

In [ ]:
for item in store.queue_items():
    print(f"{item['id']}  {item['kind']:<19} {item['reason'][:90]}")

## 4. What a person still owns

The queue: contradictions, failures with no expected answer yet, flagged tests. Resolving an item
is a decision with a record (`selfimprove queue accept|edit|reject <id>` from a terminal). An
edited test counts toward the fourth baseline number below.

In [ ]:
flagged = [i for i in store.queue_items() if i["kind"] == "flagged_test"]
if flagged:
    print(store.resolve(flagged[0]["id"], "edited", note="tightened the expected answer")["status"])
print(store.summary())

## 5. The four baseline numbers

*Part 2, section 7 and Part 3, section 1.* Traces in, themes out, tests generated, and tests a person
edited before merging. They say how much more of the loop to automate: many edits means the
criteria or the judge do not match the reviewers yet, so fix the verifier first.

In [ ]:
from selfimprove.workflows.report import report

print(report())

**Next:** [05 · Verifier and optimizer](05_verifier_and_optimizer.ipynb).